> **NOTE / PREREQUISITE:**
> 1. **Prepare Face Dataset:** Obtain or prepare your target face dataset.
> 2. **Split Large Datasets:** If the dataset is too large to fit in your available storage (Google Drive or Colab environment), split it into smaller batches or subsets using the build_segdeep_dataset.py.
> 3. **Upload to Drive:** Upload the organized dataset (or current batch) to Google Drive before running the pipeline.

In [2]:
import os
import zipfile
from pathlib import Path
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Configuration paths
ZIP_PATH = "/content/drive/MyDrive/Inpaint/swap4.zip"
DEST_DIR = "/content/muestras"

# Ensure target directory exists
Path(DEST_DIR).mkdir(parents=True, exist_ok=True)

# Extract ZIP file contents
if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(DEST_DIR)
    print(f"[INFO] ZIP successfully extracted to: {DEST_DIR}")
else:
    raise FileNotFoundError(f"[ERROR] Specified ZIP file not found at: {ZIP_PATH}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[INFO] ZIP successfully extracted to: /content/muestras


In [4]:
# Install required libraries for InsightFace and Face Parsing
!pip install -q insightface onnxruntime-gpu transformers pillow opencv-python tqdm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 762.2/762.2 kB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 202.2/202.2 MB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 38.2 MB/s eta 0:00:00


In [6]:
# ============================================================
# FACE SWAP + ORIGINAL MASK + FAKE MASK
# + SKIP X IMAGES
# + PROCESS ONLY N IMAGES
# + AUTOMATIC DOWNLOAD OF inswapper_128.onnx
# ============================================================

import json
import os
import random
import urllib.request
from datetime import datetime
from pathlib import Path

import cv2
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from tqdm import tqdm

import insightface
from insightface.app import FaceAnalysis
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

# ============================================================
# CONFIGURATION
# ============================================================

REAL_DIR = Path("/content/muestras/swap4")
BASE_OUTPUT = Path("/content/drive/MyDrive/Inpaint/muestras")

FAKE_DIR = BASE_OUTPUT / "fake_swap2"
FAKE_MASKS_DIR = BASE_OUTPUT / "masks_fake_swap2"
ORIGINAL_MASKS_DIR = BASE_OUTPUT / "masks_original2"
LOG_FILE = BASE_OUTPUT / "swap_log.json"

# Execution parameters
SKIP_COUNT = 4683  # Number of initial images to skip
NUM_IMAGES_TO_PROCESS = 10  # Number of images to process
SEED = 42

# ============================================================
# DIRECTORY CREATION & SEEDING
# ============================================================

for directory in [FAKE_DIR, FAKE_MASKS_DIR, ORIGINAL_MASKS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# DOWNLOAD SWAPPER MODEL
# ============================================================

swapper_path = os.path.expanduser("~/.insightface/models/inswapper_128.onnx")
os.makedirs(os.path.dirname(swapper_path), exist_ok=True)

if not os.path.exists(swapper_path):
    print("\nDownloading inswapper model...")
    url = "https://github.com/deepinsight/insightface/releases/download/v0.7/inswapper_128.onnx"
    urllib.request.urlretrieve(url, swapper_path)
    print("Download completed.")
else:
    print("inswapper model already exists.")

# ============================================================
# LOAD INSIGHTFACE
# ============================================================

print("\n[1/5] Initializing InsightFace...")
app = FaceAnalysis(
    name="buffalo_l",
    providers=["CUDAExecutionProvider", "CPUExecutionProvider"]
)

# Lower detection threshold to 0.25 to detect faces at difficult angles or lighting conditions
app.prepare(ctx_id=0, det_size=(640, 640), det_thresh=0.25)
swapper = insightface.model_zoo.get_model(swapper_path)
print("InsightFace loaded successfully.")

# ============================================================
# LOAD FACE PARSING MODEL
# ============================================================

print("\n[2/5] Loading Face Parsing model...")
processor = SegformerImageProcessor.from_pretrained("jonathandinu/face-parsing")
segmenter = SegformerForSemanticSegmentation.from_pretrained("jonathandinu/face-parsing").to("cuda")
print("Face Parsing loaded successfully.")

# ============================================================
# MASK GENERATION FUNCTION
# ============================================================

def generate_face_parsing_mask(image_bgr: np.ndarray) -> np.ndarray:
    """Generates a smoothed face mask from an input BGR image using Segformer."""
    rgb_image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil_image = Image.fromarray(rgb_image)

    inputs = processor(images=pil_image, return_tensors="pt").to("cuda")

    with torch.no_grad():
        outputs = segmenter(**inputs)

    logits = F.interpolate(
        outputs.logits,
        size=pil_image.size[::-1],
        mode="bilinear",
        align_corners=False
    )

    pred = logits.argmax(1)[0].cpu().numpy()

    # Target classes representing facial components
    mask = np.zeros(pred.shape, dtype=np.uint8)
    face_class_ids = [1, 2, 3, 4, 5, 10, 11, 12, 13]
    mask[np.isin(pred, face_class_ids)] = 255

    # Apply morphological dilation and Gaussian blur for smooth edges
    kernel = np.ones((21, 21), np.uint8)
    mask = cv2.dilate(mask, kernel, iterations=1)
    mask = cv2.GaussianBlur(mask, (31, 31), 0)

    return mask

# ============================================================
# LOAD IMAGES
# ============================================================

images = []
for ext in ["*.png", "*.jpg", "*.jpeg"]:
    images.extend(list(REAL_DIR.glob(ext)))

images = sorted(images)

if not images:
    raise FileNotFoundError(f"No images found in {REAL_DIR}")

print(f"\nTotal images found: {len(images)}")

# Apply skip and batch limit
images = images[SKIP_COUNT : SKIP_COUNT + NUM_IMAGES_TO_PROCESS]
print(f"Skipping: {SKIP_COUNT}")
print(f"Processing: {len(images)}")

# ============================================================
# PROCESSING LOOP
# ============================================================

print("\n[4/5] Processing images...")

log_records = []
successful_count = 0
no_face_count = 0
error_count = 0

for target_path in tqdm(images):
    fake_path = FAKE_DIR / target_path.name
    fake_mask_path = FAKE_MASKS_DIR / target_path.name
    original_mask_path = ORIGINAL_MASKS_DIR / target_path.name

    try:
        candidates = [p for p in images if p != target_path]
        if not candidates:
            continue

        source_path = random.choice(candidates)

        target_img = cv2.imread(str(target_path))
        source_img = cv2.imread(str(source_path))

        if target_img is None or source_img is None:
            error_count += 1
            continue

        target_faces = app.get(target_img)
        source_faces = app.get(source_img)

        if not target_faces or not source_faces:
            no_face_count += 1
            continue

        # Execute face swap
        result_img = swapper.get(
            target_img.copy(),
            target_faces[0],
            source_faces[0],
            paste_back=True
        )

        # Generate facial segmentation masks
        original_mask = generate_face_parsing_mask(target_img)
        fake_mask = generate_face_parsing_mask(result_img)

        # Save processed outputs
        cv2.imwrite(str(fake_path), result_img)
        cv2.imwrite(str(fake_mask_path), fake_mask)
        cv2.imwrite(str(original_mask_path), original_mask)

        successful_count += 1
        log_records.append({
            "image": target_path.name,
            "source": source_path.name,
            "status": "ok"
        })

    except Exception as e:
        error_count += 1
        log_records.append({
            "image": target_path.name,
            "status": str(e)
        })

# ============================================================
# SAVE LOGS & SUMMARY
# ============================================================

log_data = {
    "timestamp": str(datetime.now()),
    "skipped": SKIP_COUNT,
    "processed": len(images),
    "successful": successful_count,
    "no_face_detected": no_face_count,
    "errors": error_count,
    "details": log_records
}

with open(LOG_FILE, "w", encoding="utf-8") as f:
    json.dump(log_data, f, indent=4, ensure_ascii=False)

print("\n[5/5] Processing complete!")
print(f"Successful: {successful_count}")
print(f"No face detected: {no_face_count}")
print(f"Errors: {error_count}")
print(f"Saved output to: {BASE_OUTPUT}")

inswapper model already exists.

[1/5] Initializing InsightFace...
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/2d106det.onnx landmark_2d_106 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/det_10g.onnx detection [1, 3, '?', '?'] 127.5 128.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/genderage.onnx genderage ['None', 3, 96, 96] 0.0 1.0
Applied providers: ['CPUExecutionProvider'], with options: {'CPUExecutionProvider': {}}
find model: /root/.insightface/models/buffalo_l/w600k_r50.onnx recognition ['No

Loading weights:   0%|          | 0/1172 [00:00<?, ?it/s]

Face Parsing loaded successfully.

Total images found: 10000
Skipping: 4683
Processing: 10

[4/5] Processing images...


100%|██████████| 10/10 [00:46<00:00,  4.62s/it]


[5/5] Processing complete!
Successful: 7
No face detected: 3
Errors: 0
Saved output to: /content/drive/MyDrive/Inpaint/muestras
